# Projeto Final - Data Lake

## Arquitetura Medallion

### Notebook 01 - Camadas Bronze e Silver

**Objetivos**

Neste notebook serão realizadas as seguintes etapas:

- Leitura do dataset
- Exploração inicial dos dados
- Criação dos lotes
- Construção da camada Bronze
- Construção da camada Silver
- Particionamento dos dados
- Geração de logs do pipeline

Dataset utilizado: DATATRAN 2026 (PRF)

In [6]:
import pandas as pd
import os
import json

from datetime import datetime

## Configurações do Projeto

Nesta etapa definimos o caminho do dataset e criamos automaticamente as pastas do Data Lake.

In [7]:
ARQUIVO = "../../data/datatran2026.csv"

os.makedirs("../datalake/bronze", exist_ok=True)
os.makedirs("../datalake/silver", exist_ok=True)
os.makedirs("../datalake/silver/particionado", exist_ok=True)
os.makedirs("../logs", exist_ok=True)

## Leitura do Dataset

Nesta etapa os dados são carregados diretamente do arquivo CSV.

In [8]:
df = pd.read_csv(
    ARQUIVO,
    sep=";",
    encoding="latin1",
    low_memory=False
)

print("Dataset carregado com sucesso!")

df.head()

Dataset carregado com sucesso!


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,742921,2026-01-01,quinta-feira,04:04:00,TO,153,155,ARAGUAINA,Objeto estático sobre o leito carroçável,Tombamento,...,0,1,3,0,5,"-7,291548","-48,286252",SPRF-TO,DEL02-TO,UOP01-DEL02-TO
1,742942,2026-01-01,quinta-feira,06:40:00,MG,262,"146,1",RIO CASCA,Condutor Dormindo,Colisão frontal,...,4,1,1,4,3,"-20,0240733","-42,7422291",SPRF-MG,DEL03-MG,UOP03-DEL03-MG
2,742943,2026-01-01,quinta-feira,06:58:00,SC,101,193,BIGUACU,Reação tardia ou ineficiente do condutor,Colisão lateral mesmo sentido,...,0,1,2,1,4,"-27,48935222","-48,65565777",SPRF-SC,DEL01-SC,UOP01-DEL01-SC
3,742947,2026-01-01,quinta-feira,07:05:00,DF,60,23,BRASILIA,Reação tardia ou ineficiente do condutor,Colisão traseira,...,0,1,1,1,3,"-15,988927","-48,2263",SPRF-DF,DEL03-DF,UOP01-DEL03-DF
4,742960,2026-01-01,quinta-feira,06:17:00,MT,163,1044,MATUPA,Transitar na contramão,Colisão frontal,...,1,1,3,1,5,"-10,14330603","-54,93075559",SPRF-MT,DEL06-MT,UOP03-DEL06-MT


## Exploração Inicial

Verificação da estrutura do dataset.

In [9]:
print("Quantidade de linhas:", len(df))
print("Quantidade de colunas:", len(df.columns))

Quantidade de linhas: 23475
Quantidade de colunas: 30


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23475 entries, 0 to 23474
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id                      23475 non-null  int64
 1   data_inversa            23475 non-null  str  
 2   dia_semana              23475 non-null  str  
 3   horario                 23475 non-null  str  
 4   uf                      23475 non-null  str  
 5   br                      23475 non-null  int64
 6   km                      23475 non-null  str  
 7   municipio               23475 non-null  str  
 8   causa_acidente          23475 non-null  str  
 9   tipo_acidente           23475 non-null  str  
 10  classificacao_acidente  23474 non-null  str  
 11  fase_dia                23475 non-null  str  
 12  sentido_via             23475 non-null  str  
 13  condicao_metereologica  23475 non-null  str  
 14  tipo_pista              23475 non-null  str  
 15  tracado_via             23475 

In [11]:
df.isnull().sum()

id                         0
data_inversa               0
dia_semana                 0
horario                    0
uf                         0
br                         0
km                         0
municipio                  0
causa_acidente             0
tipo_acidente              0
classificacao_acidente     1
fase_dia                   0
sentido_via                0
condicao_metereologica     0
tipo_pista                 0
tracado_via                0
uso_solo                   0
pessoas                    0
mortos                     0
feridos_leves              0
feridos_graves             0
ilesos                     0
ignorados                  0
feridos                    0
veiculos                   0
latitude                   0
longitude                  0
regional                   2
delegacia                  5
uop                       10
dtype: int64

## Criação dos Lotes

O dataset será dividido em dois lotes para simular ingestões em momentos diferentes.

In [12]:
lote1 = df.sample(
    frac=0.90,
    random_state=42
)

lote2 = df.drop(
    lote1.index
)

print(lote1.shape)
print(lote2.shape)

(21128, 30)
(2347, 30)


## Simulação de Mudanças no Lote 2

Serão adicionadas novas informações para utilização posterior no Schema Evolution.

In [13]:
lote2["nova_coluna"] = "nova_info"

lote2.loc[
    lote2.index[:50],
    "classificacao_acidente"
] = "GRAVE"

lote2.loc[
    lote2.index[50:100],
    "condicao_metereologica"
] = "CHUVA"

## Camada Bronze

Nesta camada os dados permanecem praticamente iguais à origem, recebendo apenas metadados.

In [14]:
def adicionar_metadados(df, arquivo, batch):

    df = df.copy()

    df["_source_file"] = arquivo
    df["_batch_id"] = batch
    df["_ingested_at"] = datetime.now()

    return df

In [15]:
bronze1 = adicionar_metadados(
    lote1,
    ARQUIVO,
    "LOTE_1"
)

bronze2 = adicionar_metadados(
    lote2,
    ARQUIVO,
    "LOTE_2"
)

In [16]:
bronze1.to_parquet(
    "../datalake/bronze/bronze_lote1.parquet",
    index=False
)

bronze2.to_parquet(
    "../datalake/bronze/bronze_lote2.parquet",
    index=False
)

print("Bronze criada.")

Bronze criada.


## Camada Silver

Nesta etapa são realizados tratamentos e padronizações dos dados.

In [17]:
silver = pd.concat(
    [bronze1, bronze2],
    ignore_index=True
)

In [18]:
silver.columns = (
    silver.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

In [19]:
silver["data_inversa"] = pd.to_datetime(
    silver["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [20]:
silver = silver.drop_duplicates(
    subset=["id"]
)

In [21]:
silver = silver.dropna(
    subset=[
        "data_inversa",
        "uf",
        "municipio"
    ]
)

In [22]:
colunas_texto = [

    "dia_semana",

    "uf",

    "municipio",

    "causa_acidente",

    "tipo_acidente",

    "classificacao_acidente",

    "fase_dia",

    "sentido_via",

    "condicao_metereologica",

    "tipo_pista",

    "tracado_via",

    "uso_solo",

    "regional",

    "delegacia",

    "uop"

]

for coluna in colunas_texto:

    silver[coluna] = (
        silver[coluna]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [23]:
silver["ano"] = silver["data_inversa"].dt.year

silver["mes"] = silver["data_inversa"].dt.month

silver["dia"] = silver["data_inversa"].dt.day

silver["total_vitimas"] = (
    silver["mortos"] +
    silver["feridos_leves"] +
    silver["feridos_graves"]
)

silver["acidente_grave"] = (
    silver["classificacao_acidente"]
    .isin(
        [
            "GRAVE",
            "COM VÍTIMAS FATAIS"
        ]
    )
)

In [24]:
silver.to_parquet(
    "../datalake/silver/silver.parquet",
    index=False
)

print("Silver criada.")

Silver criada.


## Particionamento

Os dados serão organizados por ano e mês.

In [25]:
for (ano, mes), grupo in silver.groupby(
    ["ano", "mes"]
):

    pasta = f"../datalake/silver/particionado/ano={ano}/mes={mes}"

    os.makedirs(
        pasta,
        exist_ok=True
    )

    grupo.to_parquet(
        f"{pasta}/dados.parquet",
        index=False
    )

## Log do Pipeline

In [26]:
log = {

    "timestamp": str(datetime.now()),

    "bronze_lote1": len(bronze1),

    "bronze_lote2": len(bronze2),

    "silver": len(silver),

    "status": "SUCESSO"

}

with open(
    "../logs/pipeline_log.json",
    "w",
    encoding="utf8"
) as f:

    json.dump(
        log,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Log salvo.")

Log salvo.


# Resumo Final

Neste notebook foram implementadas:

- Leitura do dataset
- Exploração inicial
- Criação dos lotes
- Camada Bronze
- Camada Silver
- Particionamento
- Log do pipeline

Os dados tratados serão utilizados no próximo notebook para a implementação do Delta Lake, Time Travel e Schema Evolution.